# Module 09 -- Heston Stochastic Volatility Model

**Author: Djellal Djouad** -- CrossVol Research | [crossvol.com](https://crossvol.com) | ORCID [0009-0002-4911-1118](https://orcid.org/0009-0002-4911-1118)

BSM assumes constant vol. The smile proves it wrong. Heston (1993)
makes vol itself random -- the first widely adopted stochastic vol model.

On the desk, I never used Heston for live pricing -- that is what
local vol or SABR is for. But Heston builds the right intuition
for what drives the smile: spot-vol correlation, vol-of-vol, and
mean reversion. This is an introduction, not a production calibrator.

---
*License: MIT with Educational Use Clause -- see LICENSE. Not trading advice.*


In [ ]:
import numpy as np
from scipy.stats import norm
from scipy.integrate import quad
from scipy.optimize import minimize
import matplotlib.pyplot as plt


## The Heston Model

The dynamics under the risk-neutral measure:

$$dS_t = (r - q) S_t \, dt + \sqrt{v_t} \, S_t \, dW_1$$
$$dv_t = \kappa(\theta - v_t) \, dt + \xi \sqrt{v_t} \, dW_2$$
$$\text{corr}(dW_1, dW_2) = \rho$$

Parameters: $v_0$ (initial var), $\theta$ (long-run var), $\kappa$ (mean
reversion speed), $\xi$ (vol of vol), $\rho$ (spot-vol correlation).


In [ ]:
def heston_char_func(u, S, K, T, r, q, v0, theta, kappa, xi, rho):
    """
    Heston characteristic function (log-stock price).
    Using the formulation from Albrecher et al. (2007) for numerical stability.
    """
    i = complex(0, 1)

    d = np.sqrt((rho * xi * i * u - kappa)**2 + xi**2 * (i * u + u**2))
    g = (kappa - rho * xi * i * u - d) / (kappa - rho * xi * i * u + d)

    C = (r - q) * i * u * T + (kappa * theta / xi**2) * (
        (kappa - rho * xi * i * u - d) * T
        - 2 * np.log((1 - g * np.exp(-d * T)) / (1 - g))
    )

    D = ((kappa - rho * xi * i * u - d) / xi**2) * (
        (1 - np.exp(-d * T)) / (1 - g * np.exp(-d * T))
    )

    return np.exp(C + D * v0 + i * u * np.log(S))


def heston_call_price(S, K, T, r, q, v0, theta, kappa, xi, rho):
    """
    European call price under Heston via numerical integration.
    Gil-Pelaez inversion of the characteristic function.
    """
    def integrand_P1(u):
        i = complex(0, 1)
        phi = heston_char_func(u - i, S, K, T, r, q, v0, theta, kappa, xi, rho)
        phi0 = heston_char_func(-i, S, K, T, r, q, v0, theta, kappa, xi, rho)
        return np.real(np.exp(-i * u * np.log(K)) * phi / (i * u * phi0))

    def integrand_P2(u):
        i = complex(0, 1)
        phi = heston_char_func(u, S, K, T, r, q, v0, theta, kappa, xi, rho)
        return np.real(np.exp(-i * u * np.log(K)) * phi / (i * u))

    P1 = 0.5 + (1 / np.pi) * quad(integrand_P1, 1e-8, 100, limit=200)[0]
    P2 = 0.5 + (1 / np.pi) * quad(integrand_P2, 1e-8, 100, limit=200)[0]

    return S * np.exp(-q * T) * P1 - K * np.exp(-r * T) * P2


In [ ]:
# Sanity check -- low xi should converge to BSM
S, K, T, r, q = 100, 100, 0.5, 0.05, 0.0
hp = heston_call_price(S, K, T, r, q, v0=0.04, theta=0.04, kappa=5.0, xi=0.01, rho=0.0)
d1 = (np.log(S / K) + (r + 0.5 * 0.04) * T) / (0.2 * np.sqrt(T))
d2 = d1 - 0.2 * np.sqrt(T)
bsm = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
print(f"Heston (low xi): {hp:.4f}, BSM: {bsm:.4f}, diff: {abs(hp - bsm):.6f}")


## Heston Smile vs BSM Flat Vol

Negative rho creates the classic equity skew -- when spot drops,
vol rises (leverage effect). Higher xi steepens the smile.


In [ ]:
# Realistic equity parameters
v0, theta, kappa, xi, rho = 0.04, 0.04, 2.0, 0.5, -0.7
strikes = np.linspace(80, 120, 30)

def bs_call(S, K, T, r, sigma, q=0.0):
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def implied_vol_from_price(price, S, K, T, r, q=0.0):
    """Newton-Raphson IV solver."""
    sigma = 0.2
    for _ in range(100):
        p = bs_call(S, K, T, r, sigma, q)
        d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        vega = S * np.exp(-q * T) * norm.pdf(d1) * np.sqrt(T)
        if vega < 1e-12: break
        sigma = max(sigma - (p - price) / vega, 0.001)
    return sigma

heston_ivs = [implied_vol_from_price(
    heston_call_price(S, K, T, r, q, v0, theta, kappa, xi, rho), S, K, T, r, q)
    for K in strikes]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(strikes, [iv * 100 for iv in heston_ivs], 'b-o', markersize=4,
        linewidth=2, label='Heston implied vol')
ax.axhline(20, color='gray', linestyle='--', linewidth=1, label='BSM flat vol (20%)')
ax.axvline(100, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Strike', fontsize=12)
ax.set_ylabel('Implied Volatility (%)', fontsize=12)
ax.set_title(f'Heston Smile vs BSM Flat Vol (rho={rho}, xi={xi})', fontsize=14)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('../data/09_heston_smile.png', dpi=100, bbox_inches='tight')
plt.show()


## Parameter Sensitivity

Quick intuition: rho controls skew direction, xi controls curvature,
kappa controls how fast the smile flattens with maturity.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for rho_val in [-0.9, -0.5, 0.0, 0.5]:
    ivs = [implied_vol_from_price(
        heston_call_price(S, K, T, r, q, v0, theta, kappa, xi, rho_val),
        S, K, T, r, q) * 100 for K in strikes]
    axes[0].plot(strikes, ivs, linewidth=1.5, label=f'rho={rho_val}')
axes[0].set_title('Effect of rho (spot-vol correlation)')
axes[0].set_xlabel('Strike'); axes[0].set_ylabel('IV (%)'); axes[0].legend(fontsize=9)

for xi_val in [0.1, 0.3, 0.5, 0.8]:
    ivs = [implied_vol_from_price(
        heston_call_price(S, K, T, r, q, v0, theta, kappa, xi_val, rho),
        S, K, T, r, q) * 100 for K in strikes]
    axes[1].plot(strikes, ivs, linewidth=1.5, label=f'xi={xi_val}')
axes[1].set_title('Effect of xi (vol of vol)')
axes[1].set_xlabel('Strike'); axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../data/09_heston_sensitivity.png', dpi=100, bbox_inches='tight')
plt.show()


## Simple Calibration

We generate a synthetic target smile and calibrate Heston to it.
In production you would use a fast Fourier transform (Carr-Madan)
and calibrate across multiple expiries simultaneously. This is
the simple version -- one expiry, scipy minimize.


In [ ]:
# Target smile -- synthetic market quotes
target_strikes = np.array([85, 90, 95, 100, 105, 110, 115])
target_ivs = np.array([0.28, 0.25, 0.22, 0.20, 0.19, 0.185, 0.183])

# Convert target IVs to prices
target_prices = [bs_call(S, K, T, r, iv) for K, iv in zip(target_strikes, target_ivs)]

def calibration_error(params):
    v0_c, theta_c, kappa_c, xi_c, rho_c = params
    # Parameter bounds enforcement
    if v0_c <= 0 or theta_c <= 0 or kappa_c <= 0 or xi_c <= 0:
        return 1e6
    if rho_c <= -1 or rho_c >= 1:
        return 1e6
    # Feller condition check (not strict, just penalize)
    total_err = 0
    for K, target_p in zip(target_strikes, target_prices):
        try:
            model_p = heston_call_price(S, K, T, r, q, v0_c, theta_c, kappa_c, xi_c, rho_c)
            total_err += (model_p - target_p)**2
        except Exception:
            return 1e6
    return total_err

# Initial guess
x0 = [0.04, 0.04, 2.0, 0.4, -0.5]
bounds = [(0.001, 0.5), (0.001, 0.5), (0.01, 20), (0.01, 2.0), (-0.99, 0.99)]

result = minimize(calibration_error, x0, method='Nelder-Mead',
                  options={'maxiter': 5000, 'xatol': 1e-8})

v0_cal, theta_cal, kappa_cal, xi_cal, rho_cal = result.x
print(f"Calibrated: v0={v0_cal:.4f}, theta={theta_cal:.4f}, kappa={kappa_cal:.2f}, "
      f"xi={xi_cal:.4f}, rho={rho_cal:.4f}")
print(f"Feller: 2*kappa*theta={2*kappa_cal*theta_cal:.4f} vs xi^2={xi_cal**2:.4f} "
      f"{'OK' if 2*kappa_cal*theta_cal > xi_cal**2 else 'VIOLATED'}")


In [ ]:
# Plot calibrated smile vs target
cal_ivs = []
for K in strikes:
    p = heston_call_price(S, K, T, r, q, v0_cal, theta_cal, kappa_cal, xi_cal, rho_cal)
    cal_ivs.append(implied_vol_from_price(p, S, K, T, r, q) * 100)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(strikes, cal_ivs, 'b-', linewidth=2, label='Heston calibrated')
ax.plot(target_strikes, [iv * 100 for iv in target_ivs], 'ro', markersize=8,
        label='Market quotes', zorder=5)
ax.axhline(20, color='gray', linestyle='--', linewidth=0.8, label='ATM vol')
ax.set_xlabel('Strike', fontsize=12)
ax.set_ylabel('Implied Volatility (%)', fontsize=12)
ax.set_title('Heston Calibration to Synthetic Market Smile', fontsize=14)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('../data/09_heston_calibration.png', dpi=100, bbox_inches='tight')
plt.show()


## Desk Perspective

Heston captures the smile but misses jumps -- Feb 5, 2018, the VIX
doubled in an afternoon. No diffusion model predicts that.

In practice: **local vol** for exotic pricing, **SABR** for smile
interpolation, **Heston** for understanding. Bates adds jumps,
rough vol adds memory. But start here first.

---

**Next:** [Module 10 -- Risk Metrics](10_risk_metrics.py)

---
*Djellal Djouad -- CrossVol Research -- 2026*
